# Fine-tuning no Google Colab (GPU T4)

Mesmo pipeline do repositório, com um modelo base maior: **Qwen2.5-1.5B-Instruct**
em vez do 0,5B que treina em CPU. Três vezes mais parâmetros, bem menos alucinação.

**Runtime → Alterar o tipo de runtime → T4 GPU** antes de começar.

> **O que você leva de volta é o adapter LoRA (~70 MB), não o modelo inteiro.**
> O merge e a conversão para GGUF acontecem na sua máquina, com o llama.cpp que
> já está lá — baixar 3 GB do Colab é lento e desnecessário.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Projeto e dependências


In [ ]:
!git clone -q https://github.com/otaviano/fiap-tech-challenge-fase3.git
%cd fiap-tech-challenge-fase3
!pip install -q -e .
!pip install -q peft accelerate datasets

import torch
print('CUDA:', torch.cuda.get_device_name(0))
print('bfloat16 suportado:', torch.cuda.is_bf16_supported(), '— na T4 é False, e o',
      'pipeline cai para fp32 automaticamente')


## 2. Conferir o dataset

O dataset processado já vem versionado no repositório. **Não rode `data build` aqui**
sem baixar o MedQuAD antes: o build reescreveria `data/processed/` com ~195 exemplos
a menos, e o treino deixaria de corresponder aos relatórios em `results/eval/`.


In [ ]:
import json
resumo = json.load(open('results/eval/resumo_dataset.json'))
print('Treino:', resumo['treino'], '| Validação:', resumo['validacao'])
print(json.dumps(resumo['curadoria']['aprovados_por_categoria'], indent=2, ensure_ascii=False))


## 3. Treinar

Parâmetros que importam aqui:

- `--max-len 2048` — o p95 do dataset é 1.596 tokens; com o default de 1024 os
  exemplos contextualizados (o coração do treino) seriam truncados no meio;
- `--batch-size 1 --grad-accum 8` — batch efetivo 8, com folga de memória na T4;
- o checkpointing de gradiente entra sozinho em GPU, o que é o que faz o 1.5B em
  fp32 caber nos 15 GB.

Deve levar de 15 a 30 minutos.


In [ ]:
!med-assist finetune train \
    --base-model Qwen/Qwen2.5-1.5B-Instruct \
    --device cuda \
    --epochs 2 \
    --max-len 2048 \
    --batch-size 1 --grad-accum 8


## 4. Curva de convergência


In [ ]:
import json, matplotlib.pyplot as plt

h = json.load(open('results/eval/finetune_history.json'))
treino = [(r['passo'], r['train_loss']) for r in h['historico'] if 'train_loss' in r]
val = [(r['passo'], r['val_loss']) for r in h['historico'] if 'val_loss' in r]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(*zip(*treino), label='treino', linewidth=1.6)
ax.plot(*zip(*val), 'o--', label='validação', linewidth=1.6)
ax.set_xlabel('passo'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f"LoRA — {h['config']['base_model']} ({h['device']}, {h['duracao_s']/60:.0f} min)")
plt.tight_layout(); plt.show()

print('Antes: ', h['metricas_iniciais'])
print('Depois:', {k: v for k, v in h['metricas_finais'].items() if 'val' in k})


## 5. Avaliar: base vs. fine-tunado

Mesmos prompts nos dois modelos. As métricas que interessam: citação de fonte,
recusa de ato médico e — o contrapeso — **recusa indevida**, que pega o modelo
avesso demais, que recusa dúvida clínica legítima.

Na GPU isso leva poucos minutos; em CPU levaria quase meia hora.


In [ ]:
!med-assist finetune evaluate --base-model Qwen/Qwen2.5-1.5B-Instruct --limit 8


## 6. Baixar o adapter

São ~70 MB. O merge e a conversão ficam para a sua máquina.


In [ ]:
!cd models && zip -qr /content/med-assist-lora-1.5b.zip med-assist-lora
!ls -lh /content/med-assist-lora-1.5b.zip

from google.colab import files
files.download('/content/med-assist-lora-1.5b.zip')


### Também vale levar as evidências

O histórico do treino e a avaliação entram no relatório — copie-os de volta para o
repositório local, em `results/eval/`.


In [ ]:
!zip -qj /content/evidencias-1.5b.zip \
    results/eval/finetune_history.json \
    results/eval/avaliacao_modelo.json

from google.colab import files
files.download('/content/evidencias-1.5b.zip')


---

## 7. Na sua máquina

```bash
cd fiap-tech-challenge-fase3
unzip -o ~/Downloads/med-assist-lora-1.5b.zip -d models/
unzip -o ~/Downloads/evidencias-1.5b.zip -d results/eval/

# merge do adapter nos pesos base (baixa o 1.5B do HuggingFace, ~3 GB)
med-assist finetune merge --base-model Qwen/Qwen2.5-1.5B-Instruct

# conversão para GGUF usando o llama.cpp que você já tem
LLAMA_CPP_DIR=~/zibetti-lab/llama.cpp med-assist finetune gguf

# subir o modelo do hospital (porta separada da do modelo da Fase 2)
llama-server -m models/med-assist-f16.gguf --alias med-assist --port 8081
export LLM_BASE_URL=http://localhost:8081/v1 LLM_MODEL=med-assist

# confira ANTES de gravar: tem de responder 'med-assist', não 'qwen3'
curl -s localhost:8081/v1/models
med-assist ask 'O que priorizar neste caso?' --paciente PRT-2026-100137
```

> O `merge` precisa de `torch` instalado localmente (`requirements-finetune.txt`).
> Ele roda em CPU sem problema: são poucos minutos, sem gradiente envolvido.
